# Object / Label Inventory over All Sequence Folders

Scans the raw sequence root **folder by folder** (not via the manifest, so empty and
broken folders are also reported), and builds:

1. **Folder health summary** — complete / missing files / empty folders.
2. **Object inventory** — every object name from metadata JSON `Label Objects` tags and
   Label V2 `Which object` rows, with sequence counts, Label V2 class distribution,
   distance stats, and the match against `configs/datasets/object_height_catalog.csv`.

**Smoke test first:** the sequence root contains many folders. Leave `SMOKE_TEST = True`
(first `SMOKE_LIMIT` folders) to verify everything runs, then set `SMOKE_TEST = False`
for the full scan.

Script version (manifest-based, no empty-folder report): `scripts/build_object_inventory.py`.

In [ ]:
import csv
import json
import re
import sys
from collections import Counter, defaultdict
from itertools import islice
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

pd.set_option("display.max_rows", 200)
print(f"colab={IN_COLAB}")

## Config

Paths come from the repo profile (`configs/paths/*.yaml`) when the repo is available
(cloned in Colab via `scripts/bootstrap_colab.sh`, or running locally). Any path can be
overridden below.

In [ ]:
SMOKE_TEST = True    # True: scan only SMOKE_LIMIT folders to verify the code
SMOKE_LIMIT = 100
SMOKE_MODE = "random"  # "random": unbiased sample of folders (recommended - consecutive
                       # folders are usually one test campaign with the same objects);
                       # "first": first SMOKE_LIMIT folders in sorted order
SMOKE_SEED = 0
PROFILE = "colab_drive" if IN_COLAB else "local"

# Optional overrides (Path or None)
RAW_ROOT_OVERRIDE = None      # sequence root folder
OUTPUT_DIR_OVERRIDE = None    # where inventory CSVs are written
CATALOG_CSV_OVERRIDE = None   # object height catalog CSV


def find_repo_root() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    colab_clone = Path("/content/vision-uss-research")
    return colab_clone if colab_clone.exists() else None


repo_root = find_repo_root()
if repo_root:
    sys.path.insert(0, str(repo_root / "src"))
    from vision_uss_research.settings import load_paths

    paths = load_paths(profile=PROFILE)
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE or paths.outputs_dir / "profiles" / "object_inventory")
    CATALOG_CSV = Path(CATALOG_CSV_OVERRIDE or repo_root / "configs" / "datasets" / "object_height_catalog.csv")
else:
    assert RAW_ROOT_OVERRIDE and OUTPUT_DIR_OVERRIDE, "repo not found: set the *_OVERRIDE paths"
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE)
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE)
    CATALOG_CSV = Path(CATALOG_CSV_OVERRIDE) if CATALOG_CSV_OVERRIDE else None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_ROOT.is_dir(), f"raw root not found: {RAW_ROOT}"
print(f"raw root   : {RAW_ROOT}")
print(f"output dir : {OUTPUT_DIR}")
print(f"catalog    : {CATALOG_CSV}")

## Helpers

In [ ]:
LABEL_OBJECTS_SUBCATEGORY = "label objects"
CLASS_ID_TO_NAME = {"0": "traversable", "1": "low", "2": "unknown", "3": "high"}
VIDEO_EXTENSIONS = {".mp4", ".webm", ".avi"}


def normalize_object_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def load_height_catalog(catalog_csv: Path | None) -> dict[str, dict[str, str]]:
    key_to_row: dict[str, dict[str, str]] = {}
    if not catalog_csv or not catalog_csv.exists():
        print("WARNING: height catalog not found - objects will be reported as unmatched")
        return key_to_row
    with catalog_csv.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            names = [row.get("object_name", "")] + str(row.get("aliases", "")).split("|")
            for name in names:
                key = normalize_object_name(name)
                if key:
                    key_to_row.setdefault(key, row)
    return key_to_row


def safe_read_label_csv(path: Path) -> pd.DataFrame:
    attempts = [
        {"encoding": "utf-8-sig", "sep": ","},
        {"encoding": "utf-8", "sep": ","},
        {"encoding": "latin1", "sep": ","},
        {"encoding": "utf-8-sig", "sep": ";"},
        {"encoding": "latin1", "sep": ";"},
    ]
    last_error = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if len(df.columns) > 1:
                return df
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Could not read label CSV: {path}. Last error: {last_error}")


def find_column(columns: list[str], candidates: list[str]) -> str:
    norm = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        if cand.strip().lower() in norm:
            return norm[cand.strip().lower()]
    for col in columns:
        for cand in candidates:
            if cand.strip().lower() in str(col).strip().lower():
                return col
    return ""


def extract_json_tags(metadata_path: Path) -> dict[str, list[str]]:
    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
    tags: dict[str, list[str]] = defaultdict(list)
    for tag in metadata.get("tags", []):
        sub = str(tag.get("tagSubCategory", {}).get("name", "")).strip().lower()
        name = str(tag.get("name", "")).strip()
        if sub and name:
            tags[sub].append(name)
    return dict(tags)


def find_sequence_files(seq_dir: Path) -> dict:
    """Key files at the top level of a sequence folder (matches the raw data layout)."""
    top_files = [p for p in seq_dir.iterdir() if p.is_file()]
    lower = lambda p: p.name.lower()

    def pick(pred):
        matches = sorted(p for p in top_files if pred(p))
        return matches[0] if matches else None

    return {
        "n_top_files": len(top_files),
        "metadata_json": pick(lambda p: p.suffix.lower() == ".json"),
        "label_v2_csv": pick(lambda p: "pas_m_label_v2" in lower(p) and p.suffix.lower() == ".csv"),
        "front_video": pick(lambda p: "front" in lower(p) and p.suffix.lower() in VIDEO_EXTENSIONS),
        "rear_video": pick(lambda p: "rear" in lower(p) and p.suffix.lower() in VIDEO_EXTENSIONS),
        "topview": pick(lambda p: "topview" in lower(p) and p.suffix.lower() in VIDEO_EXTENSIONS),
    }


def has_any_file(seq_dir: Path, scan_cap: int = 2000) -> bool:
    """True if the folder contains at least one file anywhere (capped scan for speed)."""
    return any(p.is_file() for p in islice(seq_dir.rglob("*"), scan_cap))


def classify_folder(found: dict, is_empty: bool) -> str:
    if is_empty:
        return "empty"
    if not found["metadata_json"] and not found["label_v2_csv"]:
        return "no_metadata_no_label"
    if not found["metadata_json"]:
        return "missing_metadata_json"
    if not found["label_v2_csv"]:
        return "missing_label_v2"
    if not (found["front_video"] or found["rear_video"]):
        return "missing_camera_video"
    return "complete"

## Scan folders (smoke test: first `SMOKE_LIMIT` only)

In [ ]:
import random

all_folders = sorted(p for p in RAW_ROOT.iterdir() if p.is_dir())
print(f"total folders under raw root : {len(all_folders)}")

if SMOKE_TEST:
    if SMOKE_MODE == "random":
        rng = random.Random(SMOKE_SEED)
        folders = sorted(rng.sample(all_folders, min(SMOKE_LIMIT, len(all_folders))))
    else:
        folders = all_folders[:SMOKE_LIMIT]
else:
    folders = all_folders
print(f"folders to scan              : {len(folders)}  "
      f"(SMOKE_TEST={SMOKE_TEST}, mode={SMOKE_MODE if SMOKE_TEST else 'full'})")

In [ ]:
catalog = load_height_catalog(CATALOG_CSV)

folder_rows = []

display_name: dict[str, str] = {}
json_sequences: dict[str, set] = defaultdict(set)
label_v2_sequences: dict[str, set] = defaultdict(set)
label_v2_row_counts: Counter = Counter()
class_counts: dict[str, Counter] = defaultdict(Counter)
distances_mm: dict[str, list] = defaultdict(list)
# class values outside the documented 0-3 range -> example sequence ids
odd_class_examples: dict[str, set] = defaultdict(set)
# raw dump of every Label V2 row across all sequences (for label-file profiling)
raw_label_rows: list[dict] = []

for seq_dir in tqdm(folders, desc="Scanning folders", unit="dir"):
    sequence_id = seq_dir.name
    found = find_sequence_files(seq_dir)
    is_empty = found["n_top_files"] == 0 and not has_any_file(seq_dir)
    status = classify_folder(found, is_empty)

    json_objects: list[str] = []
    v2_objects: list[str] = []
    errors: list[str] = []

    if found["metadata_json"]:
        try:
            tags = extract_json_tags(found["metadata_json"])
            json_objects = tags.get(LABEL_OBJECTS_SUBCATEGORY, [])
        except Exception as e:
            errors.append(f"metadata_json:{type(e).__name__}")

    for name in json_objects:
        key = normalize_object_name(name)
        if key:
            display_name.setdefault(key, name)
            json_sequences[key].add(sequence_id)

    if found["label_v2_csv"]:
        try:
            df = safe_read_label_csv(found["label_v2_csv"])
            columns = list(df.columns)
            object_col = find_column(columns, ["Which object", "object"])
            class_col = find_column(columns, ["class"])
            distance_col = find_column(columns, ["important Distance [mm]", "distance"])

            for _, label_row in df.iterrows():
                rec = {"sequence_id": sequence_id}
                rec.update({str(k): label_row[k] for k in df.columns})
                raw_label_rows.append(rec)

                name = str(label_row[object_col]).strip() if object_col else ""
                if not name or name.lower() == "nan":
                    continue
                key = normalize_object_name(name)
                display_name.setdefault(key, name)
                label_v2_sequences[key].add(sequence_id)
                label_v2_row_counts[key] += 1
                v2_objects.append(name)

                if class_col:
                    class_value = str(label_row[class_col]).strip().split(".")[0]
                    if class_value and class_value.lower() != "nan":
                        class_counts[key][class_value] += 1
                        if class_value not in CLASS_ID_TO_NAME and len(odd_class_examples[class_value]) < 10:
                            odd_class_examples[class_value].add(sequence_id)

                if distance_col:
                    distance = pd.to_numeric(label_row[distance_col], errors="coerce")
                    if pd.notna(distance):
                        distances_mm[key].append(float(distance))
        except Exception as e:
            errors.append(f"label_v2:{type(e).__name__}")

    folder_rows.append({
        "sequence_id": sequence_id,
        "status": status,
        "n_top_files": found["n_top_files"],
        "has_metadata_json": int(found["metadata_json"] is not None),
        "has_label_v2_csv": int(found["label_v2_csv"] is not None),
        "has_front_video": int(found["front_video"] is not None),
        "has_rear_video": int(found["rear_video"] is not None),
        "has_topview": int(found["topview"] is not None),
        "json_label_objects": "|".join(json_objects),
        "label_v2_objects": "|".join(v2_objects),
        "errors": "|".join(errors),
    })

folders_df = pd.DataFrame(folder_rows)
print(f"scanned {len(folders_df)} folders")

## Folder health summary (incl. empty folders)

In [ ]:
status_summary = (
    folders_df["status"].value_counts().rename_axis("status").to_frame("n_folders")
)
status_summary["pct"] = (100 * status_summary["n_folders"] / len(folders_df)).round(1)
display(status_summary)

empty_df = folders_df[folders_df["status"] == "empty"]
print(f"\nEMPTY folders: {len(empty_df)}")
if len(empty_df):
    display(empty_df[["sequence_id"]])

problem_df = folders_df[~folders_df["status"].isin(["complete", "empty"])]
print(f"\nProblem folders (present but missing key files): {len(problem_df)}")
if len(problem_df):
    display(problem_df[["sequence_id", "status", "n_top_files",
                        "has_metadata_json", "has_label_v2_csv",
                        "has_front_video", "has_rear_video", "has_topview"]].head(50))

## Object inventory

The low/high bin is **derived from the Label V2 class** (the per-sequence ground truth),
not from the catalog. The catalog only contributes **metric height** where known
(staged boxes, curbs, dummy, pole) plus a prior bin; `bin_agreement` flags objects where
the catalog prior and the Label V2 majority disagree — investigate those instead of
trusting either side blindly.

In [ ]:
object_rows = []
all_keys = sorted(
    set(json_sequences) | set(label_v2_sequences),
    key=lambda k: -(len(json_sequences[k] | label_v2_sequences[k])),
)

for key in all_keys:
    catalog_row = catalog.get(key)
    dists = sorted(distances_mm[key])
    class_dist = "|".join(
        f"{CLASS_ID_TO_NAME.get(cid, cid)}:{n}"
        for cid, n in sorted(class_counts[key].items())
    )

    # Bin from Label V2: majority among the documented classes only.
    documented = {cid: n for cid, n in class_counts[key].items() if cid in CLASS_ID_TO_NAME}
    if documented:
        majority_id = max(documented, key=documented.get)
        label_v2_bin = CLASS_ID_TO_NAME[majority_id]
        label_v2_bin_purity = round(documented[majority_id] / sum(documented.values()), 3)
    else:
        label_v2_bin, label_v2_bin_purity = "", None

    catalog_bin = (catalog_row or {}).get("height_bin", "")
    if not catalog_bin or not label_v2_bin:
        bin_agreement = ""
    elif catalog_bin == "ignore":
        bin_agreement = "ok" if label_v2_bin == "traversable" else "CONFLICT"
    else:
        bin_agreement = "ok" if catalog_bin == label_v2_bin else "CONFLICT"

    object_rows.append({
        "object_name": display_name[key],
        "n_sequences_json_tag": len(json_sequences[key]),
        "n_sequences_label_v2": len(label_v2_sequences[key]),
        "n_sequences_total": len(json_sequences[key] | label_v2_sequences[key]),
        "n_label_v2_rows": label_v2_row_counts[key],
        "label_v2_class_distribution": class_dist,
        "label_v2_bin": label_v2_bin,
        "label_v2_bin_purity": label_v2_bin_purity,
        "catalog_bin_prior": catalog_bin,
        "bin_agreement": bin_agreement,
        "distance_mm_min": int(dists[0]) if dists else None,
        "distance_mm_median": int(dists[len(dists) // 2]) if dists else None,
        "distance_mm_max": int(dists[-1]) if dists else None,
        "catalog_matched": int(catalog_row is not None),
        "height_cm_min": (catalog_row or {}).get("height_cm_min", ""),
        "height_cm_max": (catalog_row or {}).get("height_cm_max", ""),
        "catalog_notes": (catalog_row or {}).get("notes", ""),
    })

objects_df = pd.DataFrame(object_rows)
display(objects_df)

conflicts_df = objects_df[objects_df["bin_agreement"] == "CONFLICT"]
if len(conflicts_df):
    print(f"\nCatalog prior vs Label V2 majority CONFLICTS: {len(conflicts_df)} - investigate:")
    display(conflicts_df[["object_name", "label_v2_class_distribution",
                          "label_v2_bin", "catalog_bin_prior"]])

unmatched_df = objects_df[objects_df["catalog_matched"] == 0]
print(f"\nObjects MISSING from the height catalog "
      f"(add to configs/datasets/object_height_catalog.csv): {len(unmatched_df)}")
if len(unmatched_df):
    display(unmatched_df[["object_name", "n_sequences_total", "label_v2_class_distribution"]])

In [ ]:
# Undocumented Label V2 class values (header documents only 0-3). Open one of the
# example sequences and inspect its label file before deciding how to map these.
if odd_class_examples:
    for class_value, examples in sorted(odd_class_examples.items()):
        total = sum(class_counts[k].get(class_value, 0) for k in class_counts)
        print(f"class value {class_value!r}: {total} label rows - example sequences:")
        for seq in sorted(examples):
            print(f"    {seq}")
else:
    print("no undocumented class values found")

In [ ]:
# Sequences available per bin (from Label V2 majority) - decides whether the
# ordinal-bin / low-vs-high design is feasible on this data.
bin_summary = (
    objects_df.assign(label_v2_bin=objects_df["label_v2_bin"].replace("", "UNLABELED"))
    .groupby("label_v2_bin")["n_sequences_total"].sum()
    .sort_values(ascending=False)
    .to_frame("n_sequences")
)
display(bin_summary)

## Label V2 file profiling (label-driven taxonomy)

Everything below reads the raw dump of all Label V2 rows collected during the scan.
Goal: understand the label file well enough to map classes to our categories by
threshold, without a hand-made object catalog.

In [ ]:
all_rows_df = pd.DataFrame(raw_label_rows)
print(f"total Label V2 rows across scanned sequences: {len(all_rows_df)}")
print(f"columns: {list(all_rows_df.columns)}")

ROW_CLASS_COL = find_column(list(all_rows_df.columns), ["class"])
ROW_OBJECT_COL = find_column(list(all_rows_df.columns), ["Which object", "object"])
ROW_DIST_COL = find_column(list(all_rows_df.columns), ["important Distance [mm]", "distance"])
ROW_NAMEOF_COL = find_column(list(all_rows_df.columns), ["NAMEoFFILE", "name of file"])

all_rows_df["_class"] = (
    all_rows_df[ROW_CLASS_COL].astype(str).str.strip().str.split(".").str[0]
)
all_rows_df["_object"] = all_rows_df[ROW_OBJECT_COL].astype(str).str.strip()

In [ ]:
# Class distribution and object x class cross-tab: is class consistent per object?
print("class value distribution (all rows):")
display(all_rows_df["_class"].value_counts().to_frame("n_rows"))

crosstab = pd.crosstab(all_rows_df["_object"], all_rows_df["_class"])
crosstab["dominant_pct"] = (100 * crosstab.max(axis=1) / crosstab.sum(axis=1)).round(1)
display(crosstab.sort_values("dominant_pct"))
print("objects with dominant_pct < 100 have inconsistent classes across sequences - inspect those rows")

In [ ]:
# Distance profile - DECISION GATE: if 'important Distance' is (nearly) constant,
# it is a labeling-spec threshold, not a measurement, and distance supervision must
# come from the CusReplay ObjDist curves instead of Label V2.
dist = pd.to_numeric(all_rows_df[ROW_DIST_COL], errors="coerce").dropna()
print(f"distance rows: {len(dist)}, unique values: {dist.nunique()}")
display(dist.value_counts().head(20).to_frame("n_rows"))

pct_2000 = 100 * (dist == 2000).mean()
print(f"rows with distance == 2000 mm: {pct_2000:.1f}%")
if pct_2000 > 90:
    print(">>> GATE: distance is a spec constant. Do NOT use Label V2 for distance "
          "supervision; use CusReplay ObjDist per cycle instead.")
else:
    dist.plot(kind="hist", bins=40, title="important Distance [mm]")

In [ ]:
# NAMEoFFILE taxonomy mining: the labeled-database path encodes the label team's own
# hierarchy, e.g. ...\D_highObjectsmedium\D12_Bicycle_stand\Type4\...
# Families observed: C_highObjectslarge, D_highObjectsmedium, E_highObjectssmall,
# F_lowObjectssmall. Categories: <letter><2 digits>_<name> (also H/J/K groups for
# surfaces/vehicles/walls). Paths come in UNC and drive-letter (W:\...) form, so match
# segments anywhere instead of anchoring on 'Labelled_Database'.
FAMILY_RE = re.compile(r"^[A-Z]_\w*[Oo]bjects\w*$")
CATEGORY_RE = re.compile(r"^[A-Z]\d{2}_")

def parse_name_of_file(value: str) -> dict:
    parts = [p for p in re.split(r"[\\/]+", str(value)) if p]
    return {
        "tax_family": next((p for p in parts if FAMILY_RE.match(p)), ""),
        "tax_category": next((p for p in parts if CATEGORY_RE.match(p)), ""),
        "tax_type": next((p for p in parts if re.match(r"^Type\d*$", p, re.I)), ""),
    }

tax = all_rows_df[ROW_NAMEOF_COL].astype(str).map(parse_name_of_file).apply(pd.Series)
all_rows_df = pd.concat([all_rows_df, tax], axis=1)

parsed_pct = 100 * (all_rows_df["tax_family"] != "").mean()
print(f"rows with a parsed D_* family: {parsed_pct:.1f}%")
display(all_rows_df["tax_family"].value_counts().to_frame("n_rows"))

# family x class: does the path family agree with the class column?
fam_class = pd.crosstab(all_rows_df["tax_family"], all_rows_df["_class"])
display(fam_class)

display(
    all_rows_df.groupby(["tax_family", "tax_category"])["sequence_id"]
    .nunique().to_frame("n_sequences").sort_values("n_sequences", ascending=False).head(40)
)

In [ ]:
# Spec-parameter columns: constant labeling-spec values or real per-row information?
for cand in ["Min exist Property", "newer than", "mpBufferAvailable", "Flag SinglePt"]:
    col = find_column(list(all_rows_df.columns), [cand])
    if col:
        vc = all_rows_df[col].value_counts(dropna=False)
        verdict = "CONSTANT (spec parameter - document and drop)" if len(vc) == 1 else f"{len(vc)} distinct values"
        print(f"{cand:<22} -> {verdict}: {dict(vc.head(5))}")

# Geometry availability
poly_col = find_column(list(all_rows_df.columns), ["PolygonX"])
if poly_col:
    has_poly = all_rows_df[poly_col].astype(str).str.strip().replace("nan", "").ne("")
    print(f"\nrows with PolygonX: {100 * has_poly.mean():.1f}%")

# Labeling campaigns
for cand in ["Date", "NT-User"]:
    col = find_column(list(all_rows_df.columns), [cand])
    if col:
        print(f"\n{cand} top values:")
        display(all_rows_df[col].value_counts().head(10).to_frame("n_rows"))

## Save outputs

In [ ]:
suffix = f"_smoke{len(folders)}" if SMOKE_TEST else ""

folders_out = OUTPUT_DIR / f"inventory_folders{suffix}.csv"
objects_out = OUTPUT_DIR / f"inventory_objects{suffix}.csv"
label_rows_out = OUTPUT_DIR / f"all_label_v2_rows{suffix}.csv"
folders_df.to_csv(folders_out, index=False)
objects_df.to_csv(objects_out, index=False)
all_rows_df.to_csv(label_rows_out, index=False)

summary_rows = [
    {"group": "overview", "value": "folders_total_under_root", "count": len(all_folders)},
    {"group": "overview", "value": "folders_scanned", "count": len(folders_df)},
    {"group": "overview", "value": "unique_objects", "count": len(objects_df)},
    {"group": "overview", "value": "objects_unmatched_in_catalog", "count": len(unmatched_df)},
]
summary_rows += [
    {"group": "folder_status", "value": status, "count": int(n)}
    for status, n in folders_df["status"].value_counts().items()
]
summary_rows += [
    {"group": "sequences_by_label_v2_bin", "value": b, "count": int(n)}
    for b, n in bin_summary["n_sequences"].items()
]
summary_rows += [
    {"group": "undocumented_class_values", "value": cv, "count": len(ex)}
    for cv, ex in sorted(odd_class_examples.items())
]
pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / f"inventory_summary{suffix}.csv", index=False)

print(f"saved:\n  {folders_out}\n  {objects_out}\n  {label_rows_out}\n  {OUTPUT_DIR / f'inventory_summary{suffix}.csv'}")

## Next steps

1. If the smoke test looks right, set `SMOKE_TEST = False` and rerun for the full scan
   (output filenames drop the `_smoke*` suffix, so smoke results are not overwritten).
2. Add every unmatched object name to `configs/datasets/object_height_catalog.csv`
   with metric height where known and a prior bin, then rerun. Remember: the catalog is
   a height/prior lookup, not ground truth - the low/high bin comes from Label V2.
3. Investigate `bin_agreement = CONFLICT` objects and undocumented class values
   (open the example sequences listed above) before locking the taxonomy.
4. Use the `sequences_by_label_v2_bin` table to judge low vs high training data volume
   for the plan in `docs/low_object_detection_plan.md`.
5. Investigate `empty` and problem folders - re-download or exclude them.